In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/lego-sets-and-themes-database/lego_sets_and_themes.csv


# 安装依赖和导入库

In [4]:
!pip install tensorflow-addons
!pip install tfa-nightly

import os

import tensorflow as tf

try:
    # 尝试让 TF 可见并使用所有物理 GPU
    gpus = tf.config.list_physical_devices('GPU')
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print("Using GPU:", gpus)
except Exception as e:
    print("GPU 初始化失败，使用 CPU。错误：", e)
    # 屏蔽 GPU
    tf.config.set_visible_devices([], 'GPU')

# 后续代码继续正常运行


from tensorflow import keras
from tensorflow.keras import layers
import tensorflow_addons as tfa


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.8/611.8 kB 10.7 MB/s eta 0:00:00 0:00:01
Using GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


# 读取并清洗数据

In [5]:
df = pd.read_csv('/kaggle/input/lego-sets-and-themes-database/lego_sets_and_themes.csv')

print(df.columns.tolist())

# 取出包含 image_url 且有套装编号（假设列名叫 'set_number'）的行
df = df.dropna(subset=['image_url', 'set_name'])
df = df.rename(columns={'Sets URL':'image_url', 'Sets Name':'set_number'})

# 映射 set_number 到连续的标签 ID
set_list = df['set_number'].unique().tolist()
label_map = {name: idx for idx, name in enumerate(set_list)}
df['label_multi'] = df['set_number'].map(label_map)
df['label_binary'] = 1  # 所有这些都是正例；后面可加负例样本

print(f"共 {len(set_list)} 种 LEGO 套装")


['set_number', 'set_name', 'year_released', 'number_of_parts', 'image_url', 'theme_name']
共 21496 种 LEGO 套装


# 构建支持 HTTPS 下载的 `tf.data.Dataset`

In [12]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def parse_row(url, lb_bin, lb_multi):
    # 利用 tf.py_function + get_file 下载并缓存，再解码
    def _download_and_decode(u):
        path = tf.keras.utils.get_file(
            fname=os.path.basename(u.numpy().decode()),
            origin=u.numpy().decode()
        )                                                  # 缓存到 ~/.keras/datasets :contentReference[oaicite:4]{index=4}
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, IMG_SIZE)
        return img

    image = tf.py_function(
        func=_download_and_decode,
        inp=[url],
        Tout=tf.float32                                  # py_function 嵌入 Python 下载逻辑 :contentReference[oaicite:5]{index=5}
    )
    image.set_shape((*IMG_SIZE, 3))
    image = image / 255.0

    return image, {'binary': lb_bin, 'multi': lb_multi}

# 构造 Dataset
ds = tf.data.Dataset.from_tensor_slices((
    df['image_url'].values,
    df['label_binary'].values,
    df['label_multi'].values
))
ds = ds.shuffle(10000, seed=42)
ds = ds.map(parse_row, num_parallel_calls=AUTOTUNE)
ds = ds.apply(tf.data.experimental.ignore_errors())
ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)               # 高效流水线 :contentReference[oaicite:6]{index=6}

# 划分训练/验证集
val_size = int(0.2 * len(df))
ds_val   = ds.take(val_size)
ds_train = ds.skip(val_size)


print("构建完毕。")

构建完毕。


# 定义多任务模型

In [13]:
# 主干网络：EfficientNetB0
base_model = keras.applications.EfficientNetB0(
    include_top=False, input_shape=(*IMG_SIZE, 3), pooling='avg'
)
base_model.trainable = False  # 先冻结预训练权重

inputs = keras.Input(shape=(*IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.Dropout(0.2)(x)

# 二分类头
bin_output = layers.Dense(1, activation='sigmoid', name='binary')(x)
# 多分类头
multi_output = layers.Dense(len(set_list), activation='softmax', name='multi')(x)

model = keras.Model(inputs=inputs, outputs=[bin_output, multi_output])

model.compile(
    optimizer='adam',
    loss= {
        'binary': 'binary_crossentropy',
        'multi':  'sparse_categorical_crossentropy'
    },
    metrics={
        'binary': ['accuracy'],
        'multi':  ['accuracy']
    }
)

model.summary()


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5             │ (None, 224, 224, 3)    │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ efficientnetb0            │ (None, 1280)           │      4,049,571 │ input_layer_5[0][0]    │
│ (Functional)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_2 (Dropout)       │ (None, 1280)           │              0 │ efficientnetb0[0][0]   │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ binary (Dense)            │ (None, 1)              │          1,281 │ dropout_2[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ multi (Dense)             │ (None, 21496)          │     27,536,376 │ dropout_2[0][0]        │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 31,587,228 (120.50 MB)

 Trainable params: 27,537,657 (105.05 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

# 训练模型

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint('best_model.keras',
                                    monitor='val_multi_accuracy',
                                    save_best_only=True),
    keras.callbacks.EarlyStopping(monitor='val_multi_accuracy',
                                  patience=5,
                                  restore_best_weights=True)
]

history = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=20,
    callbacks=callbacks
)

print("训练完毕")

Epoch 1/20
25613/25613 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
23888/23888 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
36595/36595 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
51689/51689 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
18610/18610 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
37038/37038 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/stepe
277734/277734 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
37242/37242 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
33199/33199 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
    0/38917 ━━━━━━━━━━━━━━━━━━━━ 0s 0s/stepDownloading data from https://cdn.rebrickable.com/media/sets/3072-1.jpg
21403/21403 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
38917/38917 ━━━━━━━━━━━━━━━━━━━━ 0s 2us/step
226886/226886 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
37824/37824 ━━━━━━━━━━━━━━━━━━━━ 0s 2us/step
52065/52065 ━━━━━━━━━━━━━━━━━━━━ 0s 2us/step
141356/141356 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
     0/551184 ━━━━━━━━━━━━━━━━━━━━ 0s 0s/stepm 3us/stepDownloading data from https://cdn.rebrickable.com/media/sets/7604-1.jpg
110758/110758 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
29692/29692 ━━━━━━

# 在验证集上评估

In [ ]:
# 在验证集上评估
model.evaluate(ds_val)

# 推理示例：
def predict_from_url(url):
    img = tf.keras.utils.get_file(fname=os.path.basename(url), origin=url)
    img = tf.io.read_file(img)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)[tf.newaxis,...] / 255.0
    bin_pred, multi_pred = model.predict(img)
    is_lego = (bin_pred[0][0] > 0.5)
    set_idx = np.argmax(multi_pred[0])
    return is_lego, (set_list[set_idx] if is_lego else None)

print(predict_from_url('https://cdn.rebrickable.com/media/sets/21034-1.jpg'))

from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.dataset_create_version(dataset='username/your-model-dataset',
                           files=['/kaggle/working/model.keras'],
                           version_notes='epoch20 checkpoint')

